# Does the drug matter more, or does the target matter more?

**The question this notebook answers:** when predicting how strongly a drug binds to its
protein target, how much does it help to know what the *target protein* looks like
(its sequence), versus what the *drug molecule* looks like (its chemical structure)?
And does combining both beat using either one alone?

This isn't a trick question — it's the same idea used to justify combining different
types of evidence in a lot of drug-discovery ML: proteins and small molecules are
described very differently (one is a sequence, the other is a chemical structure), so a
model that only sees one side is missing half the picture.

**How we test it:** build three simple models that predict binding affinity (how strongly
a drug and protein stick together) from:
1. The protein's sequence only (via ESM-2, a pretrained protein language model)
2. The drug's structure only (via RDKit chemical fingerprints)
3. Both together

Then compare how well each one does on data it hasn't seen. If the combined model wins,
that tells us the two sources of information are genuinely complementary — neither one
tells the whole story on its own.

## How the pieces fit together

```mermaid
flowchart LR
    A[Drug-target pair] --> B[Protein sequence]
    A --> C[Drug structure - SMILES]
    B --> D[ESM-2 embedding]
    C --> E[RDKit fingerprint]
    D --> M1[Model 1: sequence only]
    E --> M2[Model 2: structure only]
    D --> M3[Model 3: sequence + structure]
    E --> M3
    M1 --> R[Compare: which predicts binding affinity best?]
    M2 --> R
    M3 --> R
```

## Dataset

We use **Davis**, a small, well-known drug-target binding affinity benchmark (68 proteins,
442 drugs). It's small on purpose — small enough to run end to end in one Colab session,
and standard enough that anyone in the field recognizes it.

**Run this in Google Colab** (Runtime → GPU), not on a laptop CPU — the ESM-2 embedding
step is much faster with a GPU, and Colab already comes with most of what we need
preinstalled.

## Step 1 — Install the few things Colab doesn't already have

In [ ]:
%pip install -q PyTDC rdkit
print("done")

## Step 2 — Load the Davis dataset

`PyTDC` (Therapeutics Data Commons) hosts this dataset in a clean, ready-to-use form —
no manual downloading or parsing needed. We ask for the affinity values on a log scale
(`convert_to_log=True`), which is the standard way this dataset is used in the literature,
since raw binding affinities span many orders of magnitude.

In [ ]:
from tdc.multi_pred import DTI
import pandas as pd

data = DTI(name='DAVIS')
data.convert_to_log(form='binding')
split = data.get_split()

train_df = split['train']
valid_df = split['valid']
test_df = split['test']

print("train / valid / test sizes:", len(train_df), len(valid_df), len(test_df))
train_df.head()

## Step 3 — Turn each protein into a fixed-length vector (ESM-2)

ESM-2 is a protein language model: you give it an amino-acid sequence, and it gives back
a vector that captures something about that protein's biology — similar proteins get
similar vectors. We only need to embed the **unique** proteins (68 of them here), not
every row in the dataset, which keeps this step fast.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

MODEL_NAME = "facebook/esm2_t12_35M_UR50D"   # small ESM-2 checkpoint, plenty for this demo
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
esm_model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

all_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)
unique_targets = all_df[['Target_ID', 'Target']].drop_duplicates().reset_index(drop=True)
print("unique proteins to embed:", len(unique_targets))

target_embeddings = {}
with torch.no_grad():
    for _, row in unique_targets.iterrows():
        tid, seq = row['Target_ID'], row['Target']
        tokens = tokenizer(seq, return_tensors="pt", truncation=True, max_length=1024).to(device)
        out = esm_model(**tokens).last_hidden_state.mean(dim=1)  # mean-pool over the sequence
        target_embeddings[tid] = out.squeeze().cpu().numpy()

print("done embedding proteins. vector size:", len(next(iter(target_embeddings.values()))))

## Step 4 — Turn each drug into a fixed-length vector (RDKit)

A Morgan fingerprint is a standard way to represent a molecule's structure as a bit
vector — it encodes which small sub-structures (rings, functional groups, etc.) are
present. We add a few extra descriptors (molecular weight, LogP, etc.) since they're
cheap to compute and genuinely useful.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

unique_drugs = all_df[['Drug_ID', 'Drug']].drop_duplicates().reset_index(drop=True)
print("unique drugs to fingerprint:", len(unique_drugs))

def featurize_drug(smiles, n_bits=512):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
    fp_arr = np.array(fp, dtype=float)
    extra = np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol),
    ])
    return np.concatenate([fp_arr, extra])

drug_features = {}
for _, row in unique_drugs.iterrows():
    drug_features[row['Drug_ID']] = featurize_drug(row['Drug'])

print("done fingerprinting drugs. vector size:", len(next(iter(drug_features.values()))))

## Step 5 — Build the three feature sets and train three simple models

Nothing fancy here on purpose — a Random Forest for each feature set, so any difference
in performance comes from the *features*, not from one model being fancier than another.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

def build_features(df):
    seq_X, struct_X, y = [], [], []
    for _, row in df.iterrows():
        seq_X.append(target_embeddings[row['Target_ID']])
        struct_X.append(drug_features[row['Drug_ID']])
        y.append(row['Y'])
    return np.array(seq_X), np.array(struct_X), np.array(y)

seq_train, struct_train, y_train = build_features(train_df)
seq_test, struct_test, y_test = build_features(test_df)

combined_train = np.concatenate([seq_train, struct_train], axis=1)
combined_test = np.concatenate([seq_test, struct_test], axis=1)

def train_and_score(X_train, X_test, name):
    model = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r, _ = pearsonr(y_test, pred)
    print(f"{name:>16}:  RMSE = {rmse:.3f}   Pearson r = {r:.3f}")
    return {"name": name, "rmse": rmse, "pearson_r": r}

results = [
    train_and_score(seq_train, seq_test, "sequence-only"),
    train_and_score(struct_train, struct_test, "structure-only"),
    train_and_score(combined_train, combined_test, "combined"),
]

## Step 6 — Plot the comparison

In [ ]:
import matplotlib.pyplot as plt

names = [r["name"] for r in results]
rmses = [r["rmse"] for r in results]

plt.figure(figsize=(6,4))
plt.bar(names, rmses, color=["#3498db", "#e67e22", "#2ecc71"])
plt.ylabel("RMSE (lower is better)")
plt.title("Predicting binding affinity: sequence vs. structure vs. both")
plt.tight_layout()
plt.savefig("results_comparison.png", dpi=150)
plt.show()

## What this tells us

*(fill this in after running the notebook — write 2-3 sentences on which model won and
what that implies: e.g., "the combined model had the lowest RMSE, meaning the protein's
sequence and the drug's structure each carried information the other didn't — neither
alone was enough to fully explain binding strength.")*